In [ ]:
!pip install pandas nltk scikit-learn

import pandas as pd
import numpy as np
import re
import nltk

from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import accuracy_score, f1_score, classification_report

nltk.download('stopwords')
nltk.download('wordnet')

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from google.colab import files
uploaded = files.upload()

filename = list(uploaded.keys())[0]
df = pd.read_csv(filename, encoding='latin-1')

df.columns = df.columns.str.strip().str.lower()

text_col = None
for col in df.columns:
    if 'response' in col or 'feedback' in col or 'text' in col:
        text_col = col
        break

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', '', text)
    words = text.split()
    words = [lemmatizer.lemmatize(w) for w in words if w not in stop_words and len(w) > 2]
    return " ".join(words)

df['cleaned'] = df[text_col].apply(clean_text)

rules = {
    'Appreciation': ['thank','grateful','appreciate','thankful','good','great','satisfied'],
    'Challenges': ['problem','issue','delay','error','difficult','hard','slow'],
    'Suggestion': ['should','suggest','recommend','improve','better']
}

def apply_rules(text):
    labels = []
    for category, keywords in rules.items():
        for word in keywords:
            if word in text:
                labels.append(category)
                break
    if not labels:
        labels = ['Neutral']
    return labels

df['labels'] = df['cleaned'].apply(apply_rules)

mlb = MultiLabelBinarizer()
y_true = mlb.fit_transform(df['labels'])

def predict(text):
    cleaned = clean_text(text)
    labels = apply_rules(cleaned)
    bin_vec = mlb.transform([labels])[0]
    probs = bin_vec.astype(float)
    return labels, probs

def show_percentages(probs):
    total = sum(probs) if sum(probs) != 0 else 1
    for label, prob in zip(mlb.classes_, probs):
        print(f"{label}: {(prob/total)*100:.2f}%")

while True:
    x = input("Enter text (or exit): ")
    if x.lower() == 'exit':
        break
    labels, probs = predict(x)
    print("Predicted Labels:", labels)
    show_percentages(probs)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


Saving UAQTEresponses.csv to UAQTEresponses.csv
Enter text (or exit): the program really helped me graduate
Predicted Labels: ['Neutral']
Appreciation: 0.00%
Challenges: 0.00%
Neutral: 100.00%
Suggestion: 0.00%
